# 06 — Mechanism-threshold calibration
Labeled set: monitoring-relevant (IVN + adjacent) vs irrelevant. Run on server. Output = score table to pick `mechanism_edge`.

In [ ]:
import os
import yaml
try:
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer('zeroentropy/zembed-1-embedding', trust_remote_code=True)
except Exception as e:
    print('real model unavailable, using mock:', e)
    import hashlib
    import numpy as np
    class MockModel:
        """Deterministic hash-based mock embedding."""
        def encode(self, texts, normalize_embeddings=True):
            """Hash each text to a fixed unit vector."""
            V = []
            for t in texts:
                h = hashlib.md5(t.encode()).digest()
                v = np.frombuffer(h, dtype=np.uint8).astype(float)
                v = v / max(np.linalg.norm(v), 1e-9)
                V.append(v)
            return np.array(V)
    m = MockModel()
def _lens_path():
    """Pick lens yaml path for repo-root or notebooks/ cwd."""
    for c in ('config/lenses/monitoring.yaml', '../config/lenses/monitoring.yaml'):
        if os.path.exists(c):
            return c
    return 'config/lenses/monitoring.yaml'
lens = yaml.safe_load(open(_lens_path()))
probes = [p for mech in lens['mechanisms'].values() for p in (mech.get('probes', []) if isinstance(mech, dict) else [])]
print(len(probes), 'probe phrases')

In [ ]:
def _thesis():
    """Read first thesis statement from ivn scope yaml."""
    import os
    for c in ('config/scopes/ivn.yaml', '../config/scopes/ivn.yaml'):
        if os.path.exists(c):
            txt = open(c).read()
            return txt.split('thesis_statements:')[1].split('\n')[1].strip(' -"\'')
    return 'deep learning for in-vehicle CAN intrusion detection'
THESIS = _thesis()
# Reviewer-verified relevant set (REAL titles, zembed title-only mech scores as comments)
POS = [
 ('rel-drift-survey', 'A Survey on Concept Drift Adaptation'),  # Springer: Gama et al., ACM Comput Surv 2014; measured mech=0.404
 ('rel-calibration', 'On Calibration of Modern Neural Networks'),  # arXiv:1706.04599 Guo et al. 2017; measured mech=0.388
 ('rel-ensemble', 'Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles'),  # arXiv:1712.06585; measured mech=0.355
 ('rel-selective', 'Selective Classification for Deep Neural Networks'),  # arXiv:1705.08500 Geifman & El-Yaniv 2017; measured mech=0.370
 ('rel-canids', 'OTIDS: A Novel Intrusion Detection System for In-vehicle Network by Using Remote Frame'),  # IEEE Lee et al. 2017 CAN IDS; measured mech=0.490
 ('rel-nist', 'Guide to Intrusion Detection and Prevention Systems'),  # NIST SP 800-94; measured mech=0.406
]
# Controls (measured mech band 0.272-0.369 on zembed title-only)
NEG = [
 ('neg-hair', 'HairCLIP: Design Your Hair by Text and Reference Image'),  # CONTROL hairclip; arXiv:2112.05142
 ('neg-protein', 'Highly accurate protein structure prediction with AlphaFold'),  # CONTROL protein; Nature 2021 Jumper et al.
 ('neg-legal', 'Summarization of legal contracts with citation grounding'),  # CONTROL legal placeholder
 ('neg-accuracy', 'A convolutional classifier reaching 99.9% accuracy on CAN attack classification (no monitoring mechanism)'),  # CONTROL pure-accuracy CAN placeholder
 ('neg-survey', 'A survey of automotive cybersecurity standards and regulations'),  # CONTROL survey-only placeholder
]
print(len(POS), 'pos,', len(NEG), 'neg')

In [ ]:
import numpy as np
P = m.encode(['query: ' + p for p in probes], normalize_embeddings=True)
D = m.encode(['passage: ' + t for _, t in POS + NEG], normalize_embeddings=True)
T = m.encode(['query: ' + THESIS], normalize_embeddings=True)
mech = (D @ P.T).max(axis=1)
thes = (D @ T.T).ravel()
print(f"{'id':<14}{'thesis':>8}{'mech':>8}  label")
for (pid, _), th, me in zip(POS + NEG, thes, mech):
    lab = 'POS' if pid in dict(POS) else 'NEG'
    print(f'{pid:<14}{th:>8.3f}{me:>8.3f}  {lab}')
print('\npropose mechanism_edge between max-NEG-mech and min-POS-mech above')